In [ ]:
import sys, asyncio
sys.path.insert(0, "../src")

from dotenv import load_dotenv
load_dotenv("../.env.local")

from fsparts_scraper.crawler import fetch_sitemap, parse_sitemap_xml, filter_product_urls, KNOWN_CATEGORY_SLUGS
from fsparts_scraper.extractor import fetch_product_html, extract_prices, extract_images, clean_html
from fsparts_scraper.llm import get_client, extract_product_data
import random, json

llm_client = get_client()
print("NVIDIA NIM client ready")

In [ ]:
xml = fetch_sitemap()
all_urls = parse_sitemap_xml(xml)
product_urls = filter_product_urls(all_urls, KNOWN_CATEGORY_SLUGS)
print(f"Total sitemap URLs: {len(all_urls)}")
print(f"Product URLs after filtering: {len(product_urls)}")
sample = random.sample(product_urls, min(5, len(product_urls)))
for u in sample:
    print(u)

In [ ]:
# Change index to inspect different products
url = sample[0]
print(f"Inspecting: {url}\n")

html = await fetch_product_html(url)
prices = extract_prices(html)
images = extract_images(html)
cleaned = clean_html(html)

print("=== PRICES ===")
print(prices)

print("\n=== IMAGES ===")
for img in images:
    print(img)

print("\n=== LLM EXTRACTION ===")
data = extract_product_data(cleaned, client=llm_client)
print(json.dumps(data, ensure_ascii=False, indent=2))

In [ ]:
results = []
for url in sample:
    print(f"\n--- {url.split('/')[-1]} ---")
    html = await fetch_product_html(url)
    prices = extract_prices(html)
    images = extract_images(html)
    llm_data = extract_product_data(clean_html(html), client=llm_client)
    if llm_data:
        llm_data.update(prices)
        llm_data["image_urls_original"] = images
        llm_data["source_url"] = url
    results.append({"url": url, "prices": prices, "images_count": len(images), "llm": llm_data})
    print(f"  prices: {prices} | images: {len(images)} | sku: {llm_data.get('sku') if llm_data else 'FAILED'}")

print(f"\n{sum(1 for r in results if r['llm'])} / {len(results)} extractions succeeded")